# 🤝 Multi-Agent RAG System

**Day 10 — Notebook 5 of 5 | Estimated Time: 35 minutes**

---

## What You'll Learn
- Why specialised agents outperform one-size-fits-all agents
- The Orchestrator + Worker pattern
- Building a three-agent RAG pipeline: Retriever → Generator → Critic
- How agents communicate via structured messages
- Routing: the orchestrator decides which agent handles each step

---

## The Multi-Agent Pattern

```
User Query
     │
     ▼
┌─────────────────┐
│  Orchestrator   │  ← routes tasks, manages flow
└────────┬────────┘
         │ delegates to
    ┌────┴────┐
    │         │
    ▼         ▼
┌───────┐ ┌───────────┐
│Retriever│ │ Generator │  ← specialised workers
└───────┘ └─────┬─────┘
                │
                ▼
           ┌────────┐
           │ Critic │  ← quality gate
           └────────┘
                │
                ▼
           Final Answer
```

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os, json
import chromadb
from pydantic import BaseModel

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "text-embedding-004"
print("✅ Ready!")

---

## 1. Build the Knowledge Base

We'll use a corpus of AI company profiles — realistic enough to test multi-agent RAG.

In [ ]:
AI_COMPANIES = [
    {"id": "openai", "name": "OpenAI",
     "text": "OpenAI was founded in 2015 as a non-profit and converted to a capped-profit structure in 2019. It employs approximately 1,700 people as of 2024. Key products include GPT-4, ChatGPT, DALL-E, and Sora. The company raised $10 billion from Microsoft in 2023 at an implied valuation of $29 billion. Revenue was approximately $1.6 billion in 2023, growing rapidly."},
    {"id": "anthropic", "name": "Anthropic",
     "text": "Anthropic was founded in 2021 by former OpenAI researchers including Dario and Daniela Amodei. The company focuses on AI safety research and builds the Claude family of models. Anthropic raised $7.3 billion in 2023 from Google and other investors at a $18.4 billion valuation. The company employs around 500 people and is headquartered in San Francisco."},
    {"id": "deepmind", "name": "Google DeepMind",
     "text": "Google DeepMind was formed in 2023 by merging Google Brain and DeepMind. DeepMind was originally founded in London in 2010 and acquired by Google in 2014. The combined organisation employs over 4,000 researchers and engineers. Key projects include Gemini, AlphaFold, AlphaCode, and Imagen. DeepMind's AlphaFold predicted structures for over 200 million proteins."},
    {"id": "meta-ai", "name": "Meta AI",
     "text": "Meta AI is the AI research division of Meta Platforms. It publishes open-source models including LLaMA 2 and LLaMA 3, which can be downloaded and run locally. Meta AI employs approximately 2,000 researchers. The Llama 3 70B model rivals proprietary models in several benchmarks. Meta's AI research spans NLP, computer vision, and robotics."},
    {"id": "mistral", "name": "Mistral AI",
     "text": "Mistral AI is a French AI startup founded in 2023 by former Meta and DeepMind researchers. The company raised €385 million in a Series A at a €2 billion valuation. Mistral releases open-weight models including Mistral 7B and Mixtral 8x7B, which are competitive with larger closed models. The company is headquartered in Paris."},
    {"id": "cohere", "name": "Cohere",
     "text": "Cohere is a Canadian AI company founded in 2019 focused on enterprise NLP. Key products include Command (generation), Embed (embeddings), and Rerank (reranking for RAG). Cohere raised $270 million in 2023 at a $2.2 billion valuation. The company employs around 400 people. Cohere's Rerank API is widely used in production RAG systems."},
    {"id": "huggingface", "name": "Hugging Face",
     "text": "Hugging Face is an AI company and open-source platform founded in 2016. It hosts over 400,000 models and 150,000 datasets. The company raised $235 million in 2023 at a $4.5 billion valuation. Revenue comes from the Hub, the Inference API, and enterprise licensing. The transformers library has over 100,000 GitHub stars."},
    {"id": "stability", "name": "Stability AI",
     "text": "Stability AI created Stable Diffusion, one of the most popular open-source image generation models. Founded in 2020, the company raised $101 million in 2022. The company faced financial difficulties in 2024 and underwent significant leadership changes. Stable Diffusion models are widely used in creative and commercial applications."},
]

# Index in ChromaDB
chroma = chromadb.Client()
col = chroma.create_collection("ai_companies", metadata={"hnsw:space": "cosine"})

texts = [c["text"] for c in AI_COMPANIES]
result = client.models.embed_content(
    model=EMBEDDING_MODEL, contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[c["id"] for c in AI_COMPANIES],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
    metadatas=[{"name": c["name"]} for c in AI_COMPANIES],
)
print(f"✅ Indexed {col.count()} company profiles")

---

## 2. Specialised Agent Functions

Each agent is a focused function — no shared state, clean interfaces.

In [ ]:
# ─── Retriever Agent ──────────────────────────────────────────────────
def retriever_agent(query: str, top_k: int = 3) -> dict:
    """
    Specialist: semantic search over the company knowledge base.
    Returns retrieved chunks with metadata and relevance scores.
    """
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL, contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values

    results = col.query(
        query_embeddings=[q_vec], n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    chunks = [
        {
            "id": results["ids"][0][i],
            "company": results["metadatas"][0][i]["name"],
            "text": results["documents"][0][i],
            "relevance": round(1 - results["distances"][0][i], 4),
        }
        for i in range(len(results["ids"][0]))
    ]

    return {"chunks": chunks, "query": query, "retrieved": len(chunks)}


# ─── Generator Agent ──────────────────────────────────────────────────
def generator_agent(question: str, context_chunks: list[dict]) -> dict:
    """
    Specialist: generate a grounded answer from retrieved context.
    Returns the answer and which sources it drew from.
    """
    context = "\n\n".join(
        f"[{c['company']}]\n{c['text']}" for c in context_chunks
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=f"""Answer the question using ONLY the provided company profiles.
Include specific facts and figures. Cite company names when making claims.
If the answer is not in the context, say so clearly.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    sources_used = [c["company"] for c in context_chunks]
    return {"answer": response.text.strip(), "sources": sources_used}


# ─── Critic Agent ─────────────────────────────────────────────────────
class CriticVerdict(BaseModel):
    quality_score: int        # 1–5
    is_faithful: bool         # No unsupported claims
    is_complete: bool         # Addresses the question fully
    issues: list[str]         # Specific problems found
    should_regenerate: bool   # True if quality is too low
    improved_query: str       # Suggested retrieval query if regeneration needed


def critic_agent(question: str, answer: str, context_chunks: list[dict]) -> CriticVerdict:
    """
    Specialist: evaluate the generated answer for quality and faithfulness.
    Returns a verdict with pass/fail and improvement suggestions.
    """
    context_summary = "\n".join(f"- {c['company']}: {c['text'][:100]}..." for c in context_chunks)

    response = client.models.generate_content(
        model=MODEL,
        contents=f"""Evaluate this answer for quality.

Quality score (1-5):
5 = Correct, complete, well-cited, no hallucinations
4 = Good with minor gaps
3 = Adequate but missing key details or slightly vague
2 = Incomplete or has unsupported claims
1 = Wrong, hallucinated, or refuses to answer

If quality < 4, set should_regenerate=true and suggest a better retrieval query.

Context available:
{context_summary}

Question: {question}
Answer: {answer}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CriticVerdict,
            temperature=0.0,
        ),
    )
    return CriticVerdict.model_validate_json(response.text)


print("All three specialist agents defined.")

---

## 3. The Orchestrator

In [ ]:
def orchestrator(
    question: str,
    top_k: int = 3,
    max_retries: int = 2,
    min_quality: int = 4,
    verbose: bool = True,
) -> dict:
    """
    Orchestrates the three-agent pipeline:
    Retriever → Generator → Critic → (retry if needed) → Final Answer
    """
    if verbose:
        print(f"\n{'='*65}")
        print(f"Question: {question}")
        print('='*65)

    pipeline_log = []
    current_query = question

    for attempt in range(1, max_retries + 2):  # +2: first try + retries
        if verbose:
            print(f"\n[Attempt {attempt}]")

        # ── Step 1: Retrieve ──────────────────────────────────────────
        if verbose:
            print(f"  Retriever: searching for '{current_query[:50]}'")
        retrieved = retriever_agent(current_query, top_k=top_k)
        if verbose:
            for c in retrieved["chunks"]:
                print(f"    [{c['relevance']:.3f}] {c['company']}")

        # ── Step 2: Generate ─────────────────────────────────────────
        if verbose:
            print(f"  Generator: drafting answer from {len(retrieved['chunks'])} chunks")
        generated = generator_agent(question, retrieved["chunks"])
        if verbose:
            print(f"    Draft: {generated['answer'][:100]}...")

        # ── Step 3: Critique ─────────────────────────────────────────
        if verbose:
            print(f"  Critic: evaluating quality")
        verdict = critic_agent(question, generated["answer"], retrieved["chunks"])
        if verbose:
            print(f"    Score: {verdict.quality_score}/5 | Faithful: {verdict.is_faithful} | "
                  f"Complete: {verdict.is_complete}")
            if verdict.issues:
                print(f"    Issues: {verdict.issues}")

        pipeline_log.append({
            "attempt": attempt,
            "query_used": current_query,
            "chunks_retrieved": [c["company"] for c in retrieved["chunks"]],
            "quality_score": verdict.quality_score,
            "issues": verdict.issues,
        })

        # ── Accept or retry ──────────────────────────────────────────
        if verdict.quality_score >= min_quality or not verdict.should_regenerate:
            if verbose:
                print(f"  ✅ Quality accepted (score={verdict.quality_score})")
            return {
                "question": question,
                "answer": generated["answer"],
                "sources": generated["sources"],
                "quality_score": verdict.quality_score,
                "attempts": attempt,
                "pipeline_log": pipeline_log,
            }

        if attempt <= max_retries:
            if verbose:
                print(f"  ↻ Retrying with improved query: '{verdict.improved_query}'")
            current_query = verdict.improved_query

    # Return best answer even if quality threshold not met
    if verbose:
        print(f"  ⚠️ Returning best answer after {max_retries + 1} attempts")
    return {
        "question": question,
        "answer": generated["answer"],
        "sources": generated["sources"],
        "quality_score": verdict.quality_score,
        "attempts": max_retries + 1,
        "pipeline_log": pipeline_log,
    }


# Test the orchestrator
result = orchestrator("Which AI companies have raised the most funding and from whom?")
print(f"\n{'─'*65}")
print(f"FINAL ANSWER (quality={result['quality_score']}/5, {result['attempts']} attempt(s)):")
print(result["answer"])
print(f"\nSources: {result['sources']}")

In [ ]:
# Test with more questions
test_questions = [
    "What is the difference between OpenAI and Anthropic's approach to AI development?",
    "Which AI companies release open-source or open-weight models?",
    "How many employees does Google DeepMind have compared to Mistral AI?",
]

for q in test_questions:
    result = orchestrator(q, verbose=False)  # quiet mode
    print(f"\nQ: {q}")
    print(f"Score: {result['quality_score']}/5 | Attempts: {result['attempts']}")
    print(f"A: {result['answer'][:200]}...")

---

## 4. Routing — Multiple Specialised Pipelines

In [ ]:
class RouteDecision(BaseModel):
    route: str          # "rag", "factual", or "comparison"
    reasoning: str


def route_query(question: str) -> str:
    """Classify a query to determine the best pipeline."""
    response = client.models.generate_content(
        model=MODEL,
        contents=f"""Classify this question about AI companies into one routing category:

"rag"        — Requires searching the knowledge base for specific facts
"comparison" — Requires side-by-side comparison of multiple companies
"factual"    — General AI knowledge not requiring the knowledge base

Question: {question}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RouteDecision,
            temperature=0.0,
        ),
    )
    return RouteDecision.model_validate_json(response.text).route


def comparison_pipeline(question: str) -> dict:
    """Retrieve info on multiple entities and synthesise a comparison."""
    # Retrieve more chunks for comparison queries
    retrieved = retriever_agent(question, top_k=5)
    generated = generator_agent(
        f"Create a comparison table or structured analysis: {question}",
        retrieved["chunks"]
    )
    return {"answer": generated["answer"], "sources": generated["sources"], "pipeline": "comparison"}


def factual_pipeline(question: str) -> dict:
    """Answer general AI knowledge questions directly."""
    response = client.models.generate_content(
        model=MODEL, contents=question,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return {"answer": response.text.strip(), "sources": [], "pipeline": "factual"}


def smart_orchestrator(question: str, verbose: bool = True) -> dict:
    """Route → execute the appropriate pipeline → return result."""
    route = route_query(question)
    if verbose:
        print(f"  Route: {route}")

    if route == "comparison":
        return comparison_pipeline(question)
    elif route == "factual":
        return factual_pipeline(question)
    else:  # default: rag
        result = orchestrator(question, verbose=False)
        result["pipeline"] = "rag"
        return result


routing_tests = [
    "How much did Anthropic raise in 2023?",                       # → rag
    "Compare Hugging Face and Cohere in terms of focus and size",  # → comparison
    "What does the acronym LLM stand for?",                       # → factual
]

for q in routing_tests:
    print(f"\nQ: {q}")
    result = smart_orchestrator(q)
    print(f"  Pipeline used: {result['pipeline']}")
    print(f"  Answer: {result['answer'][:150]}...")

---

## 🏋️ Exercise 1: Add a Summariser Agent

For long answers, add a fourth agent that compresses the answer to a single paragraph.

In [ ]:
# Exercise 1: Summariser agent
# TODO:
# 1. Write summariser_agent(answer: str, max_sentences: int = 3) → dict
#    Use Gemini to compress the answer while preserving key facts
#
# 2. Add it to the orchestrator as an optional 4th step:
#    If the answer is > 200 words, invoke the summariser
#    Return both the full answer and the summary
#
# 3. Test on a question that typically produces a long answer:
#    "Describe the funding history and focus areas of OpenAI, Anthropic, and Google DeepMind"

def summariser_agent(answer: str, max_sentences: int = 3) -> dict:
    # TODO: Implement
    pass


# TODO: Modify the orchestrator to call summariser_agent when needed
# and test it

---

## 🏋️ Exercise 2: Agent Performance Report

Run 5 questions through the full pipeline and build a performance report.

In [ ]:
# Exercise 2: Multi-agent pipeline performance report
# TODO:
# 1. Run the following 5 questions through orchestrator()
test_questions = [
    "When was Mistral AI founded and what funding have they raised?",
    "Which companies are focused on open-source AI models?",
    "What is the employee count at Cohere vs Hugging Face?",
    "What are AlphaFold's main achievements?",
    "Which company has the largest valuation based on funding rounds?",
]

# 2. For each, record:
#    - quality_score
#    - attempts needed
#    - sources used
#    - answer length in words
#
# 3. Print a summary table:
#    Question | Score | Attempts | Sources | Words
#
# 4. Calculate: average quality score, % requiring a retry, avg sources used

print(f"{'Question':<50} {'Score':>6} {'Retries':>8} {'Sources':>8} {'Words':>6}")
print("-" * 85)
# TODO: Run evaluations and fill in the table

---

## Key Takeaways

| Component | Role | Key Design Choice |
|-----------|------|-------------------|
| Retriever Agent | Find relevant context | Focused on recall — retrieves broadly |
| Generator Agent | Produce grounded answer | Constrained to context — faithfulness first |
| Critic Agent | Quality gate | Structured verdict with retry suggestion |
| Orchestrator | Flow controller | Manages retries, calls agents in sequence |
| Router | Query classifier | Directs to the most appropriate pipeline |

**When to use multi-agent systems:**
- Different tasks need different expertise or constraints
- Quality gates are critical (e.g., legal, medical, financial)
- Parallel work can reduce latency
- Separation of concerns improves testability

---

## Day 10 Complete! 🎉

You've learned:
- ✅ Function calling: declare, call, handle results (Notebook 1)
- ✅ Building real tools: search, database, datetime (Notebook 2)
- ✅ Agent types and the ReAct pattern (Notebook 3)
- ✅ A full ReActAgent class with memory, tracing, and error handling (Notebook 4)
- ✅ Multi-agent orchestration: Retriever + Generator + Critic + Router (Notebook 5)

**Next:** Days 11–14 — Capstone Project

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [LangGraph Multi-Agent](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/multi-agent-collaboration/) | Docs | Production multi-agent framework |
| [CrewAI](https://github.com/crewAIInc/crewAI) | Library | Role-based multi-agent orchestration |
| [AutoGen](https://microsoft.github.io/autogen/) | Library | Microsoft's conversational multi-agent framework |